In [1]:
import os
import torch
from torchvision import transforms
import warnings
import json
import openpyxl as op
from torchvision import datasets
import torch.nn.functional as F
import sys
import torchvision
from tools.SpeciesClassfier_ind import SpeceseClassifier
from tools.get_sample_predict import get_sample_predict
import tools.get_size_prob
import tools.file_utils as ut
from tools.GPU_Detecter import GPU_Detect
import numpy as np
import pandas as pd 

In [2]:
notebook_path = "D:\Event\ML\size_species_match_test\Euroscaptor\test.ipynb"
__file__ = os.path.abspath(notebook_path)
sys.path.append(os.path.abspath(os.path.join(__file__, os.pardir, os.pardir)))

In [3]:
__file__

'D:\\Event\\ML\\size_species_match_test\\Euroscaptor\test.ipynb'

In [4]:
data_dir = "D:\Event\ML\size_species_match_test\Euroscaptor"
species_classfier_path = (
    "./species_classfier"
)
warnings.filterwarnings("ignore")
device = torch.device("cuda:0")
sp_classfier = SpeceseClassifier(device, species_classfier_path)  # init species classfier
print("Species Classfier Initialized Done")


cuda:0
Species Classfier Initialized Done


In [5]:
data_dir

'D:\\Event\\ML\\size_species_match_test\\Euroscaptor'

In [6]:
dataset_dir = f"{data_dir}"+"/data"
test_path = os.path.join(dataset_dir, "test")

test_transform = transforms.Compose(
    [
     transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)
dataset_dir

'D:\\Event\\ML\\size_species_match_test\\Euroscaptor/data'

In [7]:
test_dataset = datasets.ImageFolder(test_path, test_transform)
print("Test image", len(test_dataset))
print("class num", len(test_dataset.classes))
print("class name", test_dataset.classes)

Test image 60
class num 9
class name ['Euroscaptor grandis', 'Euroscaptor klossi', 'Euroscaptor kuznetsovi', 'Euroscaptor longirostris', 'Euroscaptor malayana', 'Euroscaptor micrura', 'Euroscaptor orlovi', 'Euroscaptor parvidens', 'Euroscaptor sp1']


In [8]:
json_file = open(f"{data_dir}/data/genus_labels.json", "r")
class_indict = json.load(json_file)

In [9]:
class_indict

{'0': 'Condylura cristata',
 '1': 'Desmana moschata',
 '2': 'Dymecodon pilirostris',
 '3': 'Euroscaptor',
 '4': 'Galemys pyrenaicus',
 '5': 'Mogera',
 '6': 'Neurotrichus gibbsii',
 '7': 'Orescaptor mizura',
 '8': 'Parascalops breweri',
 '9': 'Parascaptor',
 '10': 'Scalopus aquaticus',
 '11': 'Scapanulus oweni',
 '12': 'Scapanus',
 '13': 'Scaptochirus moschatus',
 '14': 'Scaptonyx',
 '15': 'Talpa',
 '16': 'Uropsilus',
 '17': 'Urotrichus talpoides'}

In [10]:
# load model
model = torchvision.models.efficientnet_b3()
model.classifier[1] = torch.nn.Linear(model.classifier[1].in_features, 18)
model = model.to(device)
weights_path = f"{data_dir}/weights/EfficientNet-B3/best_network.pth"
model.load_state_dict(torch.load(weights_path, map_location="cpu"))

<All keys matched successfully>

In [11]:
genus_data_dict = {} # key:genus, value:ind_path
for root, dir_list, file_list in os.walk(test_path):
    if len(dir_list) == 0:
        for file_name in file_list:
            i = 0
            for a in list(reversed(root.split('/'))):
                if a == 'data':
                    break
                else:
                    i += 1
            genus_name = "Euroscaptor"
            ind_name = root.split('/')[-1]
            ind_path = root
            if genus_name not in genus_data_dict.keys():
                genus_data_dict[genus_name] = []
        genus_data_dict[genus_name].append(ind_path)

model.eval()

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 40, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(40, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(40, 40, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=40, bias=False)
            (1): BatchNorm2d(40, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(40, 10, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(10, 40, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActiv

In [12]:
genus_predict_dict = {}
for genus_name in genus_data_dict.keys():
    for ind_path in genus_data_dict[genus_name]:
        # 获取该个体的尺寸信息
        
        # 修改get_sample_predict调用，传递尺寸信息
        predict_cla = get_sample_predict(ind_path, model, device, 18,True)
        predict_genus = class_indict[str(predict_cla)]
        if predict_genus not in genus_predict_dict.keys():
            genus_predict_dict[predict_genus] = [ind_path]
        else:
            genus_predict_dict[predict_genus].append(ind_path)
genus_predict_dict

{'Euroscaptor': ['D:\\Event\\ML\\size_species_match_test\\Euroscaptor/data\\test\\Euroscaptor grandis\\Euroscaptor grandis_57111_F_sichuan_chengdu',
  'D:\\Event\\ML\\size_species_match_test\\Euroscaptor/data\\test\\Euroscaptor klossi\\Euroscaptor klossi_MCZ38251_Laos',
  'D:\\Event\\ML\\size_species_match_test\\Euroscaptor/data\\test\\Euroscaptor klossi\\Euroscaptor klossi_NMNH261090_M_Thailand_Doi pu Kha_questionmark',
  'D:\\Event\\ML\\size_species_match_test\\Euroscaptor/data\\test\\Euroscaptor kuznetsovi\\Euroscaptor kuznetsovi_12522',
  'D:\\Event\\ML\\size_species_match_test\\Euroscaptor/data\\test\\Euroscaptor longirostris\\Euroscaptor longirostris_0905172_Sichuan',
  'D:\\Event\\ML\\size_species_match_test\\Euroscaptor/data\\test\\Euroscaptor longirostris\\Euroscaptor longirostris_AMNH110505_F_China_Sichuan_Qionglaishan_wenchuan',
  'D:\\Event\\ML\\size_species_match_test\\Euroscaptor/data\\test\\Euroscaptor longirostris\\Euroscaptor longirostris_FMNH45899_F_China_Sichuan_Goan

In [13]:
json_file = open(f"{data_dir}/data/more_species_labels.json", "r")
Mogera_indict = json.load(json_file)

In [14]:
import torchvision.models as models
model2_Mogera = models.efficientnet_b3()
model2_Mogera.classifier[1] = torch.nn.Linear(
    model2_Mogera.classifier[1].in_features, 9
)
model2_Mogera.to(device)
weights_Mogera_path = "D:\Event\ML\size_species_match_test\Euroscaptor\species_classfier\weights\EB3_Euroscaptor\\best_network.pth"
model2_Mogera.load_state_dict(
    torch.load(weights_Mogera_path, map_location="cpu")
)
model2_Mogera.eval()
ind_path = genus_data_dict["Euroscaptor"]

In [15]:
img_prob_all = []
for ind in ind_path:
    with torch.no_grad():
        img_prob = get_sample_predict(ind,model2_Mogera,device,9,False)
        img_prob_all.append(img_prob.numpy())
    

In [16]:
for a in img_prob_all:
    #print(a[0])
    print(Mogera_indict["Euroscaptor"][np.argmax(a[0])])

Euroscaptor grandis
Euroscaptor klossi
Euroscaptor parvidens
Euroscaptor kuznetsovi
Euroscaptor longirostris
Euroscaptor longirostris
Euroscaptor longirostris
Euroscaptor longirostris
Euroscaptor longirostris
Euroscaptor longirostris
Euroscaptor parvidens
Euroscaptor malayana
Euroscaptor longirostris
Euroscaptor micrura
Euroscaptor micrura
Euroscaptor klossi
Euroscaptor sp1
Euroscaptor orlovi
Euroscaptor parvidens
Euroscaptor sp1


In [17]:
from tools.get_size_prob_v2_pic_vote import predict_species_probability
size_model = "Euroscaptor_train_per_sample/模型文件/随机森林_weighted_vote_模型组件.pkl"
excel_extensions = ('xlsx')
# 遍历文件夹，只保留图片文件
for genus_name in genus_data_dict.keys():
    size_path = []
    for ind_excel in genus_data_dict[genus_name]:
        samples = os.listdir(ind_excel)
        for file in samples:
            if file.lower().endswith(excel_extensions):
                size_path.append(ind_excel+"\\"+file)

#size_model_config = load_size_model(size_model)
size_prob_all_df = []
for a in size_path:
    size_df=pd.read_excel(a)
    size_prob,_ = predict_species_probability(size_df,size_model)
    size_prob_all_df.append(size_prob)


加载模型成功: RandomForestClassifier
物种类别: ['Euroscaptor grandis' 'Euroscaptor klossi' 'Euroscaptor kuznetsovi'
 'Euroscaptor longirostris' 'Euroscaptor malayana' 'Euroscaptor micrura'
 'Euroscaptor orlovi' 'Euroscaptor parvidens' 'Euroscaptor sp1']
训练时特征列: ['length', 'width', 'aspect_ratio', 'area', 'perimeter', 'size_diff', 'inv_aspect_ratio', 'area_perimeter_ratio', 'length_square_over_area', 'width_ratio', 'log_length', 'log_width', 'log_area', 'size_ratio', 'shape_complexity', 'normalized_length', 'normalized_width', 'view_#m#l', 'view_#s#d', 'view_#s#l', 'view_#s#v']
正在进行数据预处理...
数据预处理完成：4个样本
特征数量：21
特征顺序：['length', 'width', 'aspect_ratio', 'area', 'perimeter', 'size_diff', 'inv_aspect_ratio', 'area_perimeter_ratio', 'length_square_over_area', 'width_ratio', 'log_length', 'log_width', 'log_area', 'size_ratio', 'shape_complexity', 'normalized_length', 'normalized_width', 'view_#m#l', 'view_#s#l', 'view_#s#v', 'view_#s#d']
标准化前数据形状: (4, 21)
特征名: ['length', 'width', 'aspect_ratio', 'area'

In [18]:
prob_columns = [col for col in size_prob.columns if '_概率' in col]
species_names = [col.replace('_概率', '') for col in prob_columns]

print(f"找到 {len(prob_columns)} 个物种的概率列")
print(f"物种: {species_names}")

找到 9 个物种的概率列
物种: ['Euroscaptor grandis', 'Euroscaptor klossi', 'Euroscaptor kuznetsovi', 'Euroscaptor longirostris', 'Euroscaptor malayana', 'Euroscaptor micrura', 'Euroscaptor orlovi', 'Euroscaptor parvidens', 'Euroscaptor sp1']


In [19]:
size_prob_all = []
for size_prob in size_prob_all_df:
    size_prob_all.append(size_prob[prob_columns].values)
size_prob_all

[array([[0.81805556, 0.01316176, 0.07336601, 0.0251634 , 0.00213235,
         0.02156046, 0.03478758, 0.01177288, 0.        ]]),
 array([[0.        , 0.1046657 , 0.00070806, 0.1583055 , 0.11271951,
         0.        , 0.04810158, 0.00441396, 0.57108568]]),
 array([[0.        , 0.41235656, 0.00235656, 0.35079918, 0.1103791 ,
         0.00144467, 0.0577459 , 0.04084016, 0.02407787]]),
 array([[0.79403074, 0.        , 0.10258269, 0.03647025, 0.        ,
         0.01509266, 0.04318056, 0.00864309, 0.        ]]),
 array([[0.0139881 , 0.12147321, 0.09931724, 0.40577381, 0.00269345,
         0.12599702, 0.14678395, 0.08397321, 0.        ]]),
 array([[0.        , 0.18576983, 0.07507375, 0.21823963, 0.33362286,
         0.01214984, 0.06451105, 0.02238905, 0.08824399]]),
 array([[0.   , 0.005, 0.005, 0.7  , 0.235, 0.04 , 0.015, 0.   , 0.   ]]),
 array([[0.01372818, 0.05006129, 0.04972354, 0.58042226, 0.02391754,
         0.07507938, 0.00648575, 0.06704056, 0.1335415 ]]),
 array([[0.00192878, 0

In [39]:
from tools.get_size_prob import predict_size_prob
import pickle
def load_size_model(model_path):
    """加载训练好的尺寸分类模型组件"""
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"尺寸模型文件不存在：{model_path}")
    with open(model_path, "rb") as f:
        model_components = pickle.load(f)
    return {
        "model": model_components["best_model"],
        "scaler": model_components["scaler"],
        "label_encoder": model_components["label_encoder"],
        "feature_selector": model_components["feature_selector"],
        "global_view_means": model_components["global_view_means"],
        "global_stats": model_components["global_stats"],
        "view_order": model_components["view_order"],
        "all_feature_columns": model_components["all_feature_columns"],
        "class_names": model_components["label_encoder"].classes_
    }
size_model_conf = load_size_model("size_part/Euroscaptor_train_enhanced_V2/模型文件/K近邻_模型组件.pkl")
size_prob_all_concat = []
for a in size_path:
    size_df=pd.read_excel(a)
    size_prob = predict_size_prob(size_df,size_model_conf)
    size_prob_all_concat.append(size_prob)

In [48]:
arr1 = np.array(size_prob_all_concat,dtype=np.float32)
result = []
for i in range(0,len(arr1)):
    #result.append(arr1[i]*np.max(arr1[i])+img_prob_all[i][0]*np.max(img_prob_all[i][0]))
    #result.append(arr1[i]+img_prob_all[i][0])
    result.append(arr1[i]*0.4+img_prob_all[i][0]*0.6)
result

[array([7.0452797e-01, 6.4172596e-06, 2.6510281e-03, 1.3729019e-01,
        1.1448263e-05, 1.4624126e-01, 3.7575126e-05, 8.2380593e-06,
        9.2259636e-03], dtype=float32),
 array([2.71597912e-07, 5.84917486e-01, 3.02421512e-07, 1.39886856e-01,
        1.17377855e-01, 2.39538145e-03, 1.24886001e-05, 6.44017290e-03,
        1.48969188e-01], dtype=float32),
 array([1.3230153e-06, 3.7947148e-01, 6.6225747e-10, 3.5574925e-01,
        1.0091186e-02, 1.4095864e-03, 1.3714961e-06, 2.5327554e-01,
        2.7622866e-07], dtype=float32),
 array([4.0000039e-01, 1.6763090e-07, 5.9998596e-01, 1.9555897e-09,
        9.3616581e-09, 2.6839169e-07, 6.6270115e-07, 1.2578131e-05,
        2.8284083e-10], dtype=float32),
 array([8.1094414e-02, 1.3553612e-01, 9.9454322e-05, 6.4226294e-01,
        2.0419188e-07, 3.2575012e-03, 1.7965437e-04, 1.3384499e-01,
        3.7247506e-03], dtype=float32),
 array([2.3077404e-05, 4.8646948e-04, 3.7070447e-05, 5.9442878e-01,
        1.2665105e-01, 7.5506315e-02, 1.207

In [22]:
json_file = open(f"{data_dir}/data/more_species_labels.json", "r")
Mogera_indict = json.load(json_file)

In [49]:
a = 0
num = 0
for i in result:
    #print(ind_path[a])
    #print(Mogera_indict["Mogera"][np.argmax(i)])
    if Mogera_indict["Euroscaptor"][np.argmax(i)] != ind_path[a].split("\\")[-2]:
        num += 1
    a+=1
num

5